# Chapter 2 — Prerequisite concepts

Seven concepts are enough to understand the paper. They build on one another in this order:

```
triples & identifiers ──► knowledge graph ──► SPARQL label query ─┐
                                                                  ├─► entity linking = candidates + disambiguation
WordNet synsets ──► lemmas (synonyms) ──► taxonomy & path length ─┘
string edit distance ───────────────────────────────────────────────► ranking names
hit@k ──────────────────────────────────────────────────────────────► evaluation
```

For each: intuition, definition, a worked example, and what it is used for later.

In [1]:
# Locate the research/ directory and make the real implementation importable.
import sys, json
from pathlib import Path
RESEARCH = Path.cwd().resolve()
while not (RESEARCH / "pel").is_dir():
    RESEARCH = RESEARCH.parent
for path in (RESEARCH, RESEARCH / "experiments"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
FIGURES = RESEARCH / "experiments" / "figures"
RESULTS = RESEARCH / "experiments" / "results"
%matplotlib inline

## 2.1 Triples, identifiers, knowledge graphs

**Intuition.** A knowledge graph is a pile of three-word sentences: *subject — predicate — object*.

**Definition [paper, Sec. 3.2, after Hogan et al.].** A knowledge graph is $G_t = (N_t, L_t)$ with nodes $N_t$ and links $L_t \subseteq N_t \times P_t \times N_t$, where $P_t$ is a set of properties.

**Identifiers.** Nodes are named by IRIs, not by words. `http://www.wikidata.org/entity/Q13191` is the fruit; `…/Q39338` is the colour. Words are attached to nodes as *label* literals, and many nodes can carry the same label. That one-to-many relation between strings and nodes is the whole difficulty of linking.

In [2]:
# A five-triple knowledge graph, enough to show the problem.
toy_graph = {
    ("Q13191", "label", "orange"), ("Q13191", "subclass of", "citrus fruit"),
    ("Q39338", "label", "orange"), ("Q39338", "instance of", "color"),
    ("Q491350", "label", "Orange"), ("Q491350", "instance of", "city in the United States"),
}
def entities_labelled(text):
    return sorted(s for s, p, o in toy_graph if p == "label" and o == text)

print("orange ->", entities_labelled("orange"))
print("Orange ->", entities_labelled("Orange"), "   (exact matching is case-sensitive)")

orange -> ['Q13191', 'Q39338']
Orange -> ['Q491350']    (exact matching is case-sensitive)


## 2.2 Asking a graph a question: SPARQL

**Intuition.** A SPARQL query is a triple with holes (`?variables`); the database returns every way of filling them.

The paper uses four query shapes [paper, Sec. 5.1]. The paper describes them in words; the SPARQL below is this project's rendering (`pel/knowledge_graphs.py`):

In [3]:
from pel.knowledge_graphs import build_retrieval_query, QUERY_TYPES
for query_type in QUERY_TYPES:
    print(f"{query_type:8s}", build_retrieval_query("orange", query_type, limit=50), "\n")

S        SELECT DISTINCT ?entity WHERE { ?entity ?anyProperty "orange"@en } LIMIT 50 

S_links  SELECT ?entity (COUNT(?target) / COUNT(DISTINCT ?anyProperty) AS ?links) WHERE { ?entity ?anyProperty "orange"@en . ?entity ?outgoing ?target } GROUP BY ?entity ORDER BY DESC(?links) LIMIT 50 

L        SELECT DISTINCT ?entity WHERE { ?entity <http://www.w3.org/2000/01/rdf-schema#label> "orange"@en } LIMIT 50 

L_links  SELECT ?entity (COUNT(?target) AS ?links) WHERE { ?entity <http://www.w3.org/2000/01/rdf-schema#label> "orange"@en . ?entity ?outgoing ?target } GROUP BY ?entity ORDER BY DESC(?links) LIMIT 50 



* `S` (“simple”): any property whose value is the string — labels, aliases, but also unrelated literals.
* `L` (“label”): only `rdfs:label`.
* `…_links`: additionally count the entity's outgoing triples and sort by that count. The count is a cheap **popularity prior**: well-known things have more statements.

## 2.3 Entity linking in two steps

**Definition [Balog 2018; Shen et al. 2015].** Given a *mention* (a string) and a knowledge graph, return the entity the mention refers to, or NIL.

Practically every system splits this into

1. **candidate generation** — a high-recall step producing a short list;
2. **disambiguation** — a ranking step that picks one (or none).

The paper keeps exactly this structure and calls the steps $f_c$ and $f_d$.

## 2.4 WordNet: senses, synonyms, a taxonomy

**Intuition.** WordNet is a dictionary organised by *meaning*. Each meaning is a **synset** (synonym set); the words in it are its **lemmas**; noun synsets are arranged in an *is-a* hierarchy (**hypernyms** above, **hyponyms** below).

In [4]:
from pel.wordnet_access import get_synsets, get_names
for synset in get_synsets("orange"):
    print(f"{synset.name():14s} {get_names(synset)!s:28s} {synset.definition()[:60]}")

orange.n.01    ['orange']                   round yellow to orange fruit of any of several citrus trees
orange.n.02    ['orange', 'orangeness']     orange color or pigment; any of a range of colors between re
orange.n.03    ['orange', 'orange tree']    any citrus tree bearing oranges
orange.n.04    ['orange']                   any pigment producing the orange color
orange.n.05    ['Orange', 'Orange River']   a river in South Africa that flows generally westward to the
orange.s.01    ['orange', 'orangish']       of the color between red and yellow; similar to the color of


In [5]:
# The is-a chain above the fruit sense: this is the "taxonomy" the method measures distances in.
fruit_sense = get_synsets("orange")[0]
print(" -> ".join(s.name().split(".")[0] for s in reversed(fruit_sense.hypernym_paths()[0])))

orange -> citrus -> edible_fruit -> fruit -> reproductive_structure -> plant_organ -> plant_part -> natural_object -> whole -> object -> physical_entity -> entity


**Used for.** Two things: (1) *synonyms* — the lemmas of a label's synsets give alternative strings to look up (“couch” → “sofa”); (2) *type checking* — the distance between two synsets in the hierarchy says how related two concepts are.

**Path similarity.** For synsets $s, t$ let $d(s,t)$ be the number of edges on the shortest path between them in the hierarchy. Then

$$\mathrm{sim}_{path}(s,t) = \frac{1}{1 + d(s,t)} \in (0, 1].$$

It is 1 for identical synsets, ½ for parent and child, ⅓ for siblings or grandparents.

In [6]:
from pel.wordnet_access import path_similarity
citrus, color, city = get_synsets("citrus")[0], get_synsets("color")[0], get_synsets("city")[0]
for other in (fruit_sense, citrus, color, city):
    d = fruit_sense.shortest_path_distance(other)
    print(f"orange.n.01 vs {other.name():12s}  d = {d:2d}   similarity = {path_similarity(fruit_sense, other):.3f}")

orange.n.01 vs orange.n.01   d =  0   similarity = 1.000
orange.n.01 vs citrus.n.01   d =  1   similarity = 0.500
orange.n.01 vs color.n.01    d = 13   similarity = 0.071
orange.n.01 vs city.n.01     d = 14   similarity = 0.067


## 2.5 Edit distance between strings

**Intuition.** How many keystrokes turn one word into another?

**Definition.** The Levenshtein distance $\mathrm{lev}(a,b)$ is the minimum number of single-character insertions, deletions and substitutions that transform $a$ into $b$. It is computed by filling a table $D$ where $D[i][j]$ is the distance between the first $i$ characters of $a$ and the first $j$ of $b$ (Chapter 4 derives the recurrence).

**Used for.** Ordering the alternative names of a label: names that look like the label come first.

In [7]:
from pel.text_similarity import levenshtein_distance, levenshtein_similarity
for a, b in [("orange", "Orange"), ("orange", "orangeness"), ("couch", "sofa"), ("skis", "ski")]:
    print(f"{a!r:10} {b!r:14} distance {levenshtein_distance(a, b)}   similarity {levenshtein_similarity(a, b):.2f}")

'orange'   'Orange'       distance 1   similarity 0.83
'orange'   'orangeness'   distance 4   similarity 0.60
'couch'    'sofa'         distance 4   similarity 0.20
'skis'     'ski'          distance 1   similarity 0.75


## 2.6 Scoring a ranked list: hit@k

A linker returns a ranked list per label. **hit@k** is the fraction of labels whose correct entity appears among the first $k$ results. hit@1 is plain accuracy; hit@3 and hit@5 tell us whether the right answer was *nearly* found (a ranking problem) or not found at all (a retrieval problem).

In [8]:
from pel.evaluation import first_hit_rank, hit_at_k, wilson_interval
truth = {"Q13191"}
rankings = [["Q13191", "Q39338"], ["Q491350", "Q13191"], ["Q39338"], []]
ranks = [first_hit_rank(r, truth) for r in rankings]
print("rank of the correct entity per label:", ranks)
print({f"hit@{k}": hit_at_k(ranks, k) for k in (1, 3, 5)})
print("95% interval for 55 correct out of 80:", tuple(round(x, 3) for x in wilson_interval(55, 80)))

rank of the correct entity per label: [1, 2, None, None]
{'hit@1': 0.25, 'hit@3': 0.5, 'hit@5': 0.5}
95% interval for 55 correct out of 80: (0.579, 0.778)


The last line previews a point that matters throughout: with only 80 labels, an accuracy of 0.69 is compatible with anything between roughly 0.58 and 0.78. Differences of a few points on such a sample mean nothing.

## 2.7 Putting the pieces together

| Concept | Role in the method |
|---|---|
| label literal → many nodes | the reason candidates must be generated and ranked |
| SPARQL label queries (S, L) | turn a string into candidate nodes |
| outgoing-link count | popularity prior (`_links` queries) |
| WordNet lemmas | more strings to try (Algorithm 1) |
| Levenshtein similarity | order of those strings (Algorithm 1) |
| WordNet path similarity | does the candidate's class fit the label? (Algorithm 2) |
| hit@k | evaluation |

**Next:** Chapter 3 reads the paper with these in hand.